# Models, Prompts and Output Parsers in LangChain

## Goal of this notebook

Large Language Models can generate text, but building reliable applications requires more structure than simply sending prompts.

In this notebook we explore the three core components used in LangChain:

1. **Models** – the interface to LLM providers
2. **Prompt Templates** – reusable prompt structures
3. **Output Parsers** – tools for converting LLM responses into structured data

We will first make direct API calls to OpenAI, and then see how LangChain simplifies the process of building LLM pipelines.

## Environment Setup

First we configure the environment and load the required libraries.

We will use:

- OpenAI API
- LangChain
- LangChain OpenAI integrations

We also load the API key from environment variables using `dotenv`.

In [45]:
import os
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

client = OpenAI()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [46]:
# Use one current chat model throughout the notebook.
# Keeping this in a single variable makes future updates simple.
llm_model = "gpt-4.1-mini"

## Direct API Calls to OpenAI

Before using LangChain, let's see how to interact directly with the OpenAI API.

We define a helper function that sends a prompt to the model and returns the generated response.

This illustrates the basic workflow of interacting with an LLM.

In [47]:
def get_completion(prompt: str, model: str = llm_model) -> str:
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0,
    )
    return response.output_text

In [4]:
get_completion("What is 1+3?")

'1 + 3 = 4'

In [5]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [6]:
style = """American English \
in a calm and respectful tone
"""

### Prompt Engineering Example

Here we construct a prompt that asks the model to translate customer text into a specific style.

This example shows how prompt design can control the tone and behavior of the model.

In [7]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [8]:
response = get_completion(prompt)

In [9]:
response

'I’m quite upset that my blender lid came off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn’t cover the cost of cleaning up the mess. I would really appreciate your assistance with this.'

## Using LangChain

While direct API calls work, they can become difficult to manage in complex applications.

LangChain provides abstractions that make it easier to:

- manage prompts
- connect models to pipelines
- process outputs

## Chat API : LangChain

Let's try how we can do the same using LangChain.

In [ ]:
# import sys
# !{sys.executable} -m pip install -U langchain-core langchain-openai openai python-dotenv

### Model

LangChain provides a wrapper for chat models such as OpenAI's GPT models.

The `ChatOpenAI` class allows us to configure the model and control parameters like temperature.

In [10]:
from langchain_openai import ChatOpenAI

In [11]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(temperature=0.0, model=llm_model)
chat

ChatOpenAI(profile={'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x1147b8f10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1147b9f10>, root_client=<openai.OpenAI object at 0x1147b8cd0>, root_async_client=<openai.AsyncOpenAI object at 0x1147b9a50>, model_name='gpt-4.1-mini', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True)

### Prompt Templates

Prompt templates allow us to create reusable prompts with variables.

Instead of writing the full prompt each time, we define a template and dynamically insert values.

This makes prompts easier to maintain and reuse.

In [12]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_template(template_string)
output_parser = StrOutputParser()

In [14]:
print(prompt_template)

input_variables=['style', 'text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n'), additional_kwargs={})]


In [15]:
print(prompt_template.input_variables)

['style', 'text']


In [16]:
customer_style = """American English \
in a calm and respectful tone
"""

In [17]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [18]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [19]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [20]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


### Building an LLM Pipeline

LangChain allows us to build pipelines using the **LCEL (LangChain Expression Language)**.

A typical pipeline consists of:

Prompt Template → Model → Output Parser

This structure makes LLM applications modular and composable.

In [21]:
# Build the standard LCEL pipeline: prompt -> model -> string parser
translation_chain = prompt_template | chat | output_parser
customer_response = translation_chain.invoke(
    {"style": customer_style, "text": customer_email}
)

In [22]:
print(customer_response)

I’m quite upset that my blender lid came off and splattered smoothie all over my kitchen walls. To make things worse, the warranty doesn’t cover the cost of cleaning up the mess. I would appreciate your assistance with this matter.


In [23]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [24]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [25]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [26]:
service_response = translation_chain.invoke(
    {"style": service_style_pirate, "text": service_reply}
)
print(service_response)

Ahoy there, valued matey! I be afeared the warranty don’t cover the cost o’ cleanin’ yer galley, for ’tis yer own folly that ye misused yer blender by forgettin’ to secure the lid afore settin’ sail with it. ’Tis a hard tide to weather, but such be the way o’ the sea! Fair winds to ye!


## Output Parsers

LLMs usually return plain text responses.

However, many applications require structured data such as JSON or Python dictionaries.

Output parsers allow us to convert raw LLM responses into structured formats that can be used programmatically.

In [27]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [28]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer true if yes, otherwise false.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price,
and output them as a JSON list of strings.

Return valid JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else?\nAnswer true if yes, otherwise false.\n\ndelivery_days: How many days did it take for the product to arrive?\nIf this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,\nand output them as a JSON list of strings.\n\nReturn valid JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [30]:
chat = ChatOpenAI(model=llm_model, temperature=0)
review_chain = prompt_template | chat | StrOutputParser()

response = review_chain.invoke({"text": customer_review})
print(response)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}
```


In [31]:
type(response)

langchain_core.messages.base.TextAccessor

In [32]:
# Parse the model output first, then access values like a normal dictionary.
JsonOutputParser().invoke(response).get("gift")

True

### Parsing Structured Outputs

In this example we ask the model to extract structured information from a customer review.

The response will include fields such as:

- whether the item was a gift
- delivery time
- price sentiment

The `JsonOutputParser` then converts the model output into a Python dictionary.

In [33]:
from langchain_core.output_parsers import JsonOutputParser

In [34]:
# `JsonOutputParser` keeps the example lightweight and works cleanly with LCEL.
output_parser = JsonOutputParser()

In [35]:
format_instructions = output_parser.get_format_instructions()

In [36]:
print(format_instructions)

Return a JSON object.


In [37]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer true if yes, otherwise false.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price,
and output them as a JSON list of strings.

Return valid JSON with the following keys:
gift
delivery_days
price_value

text: {text}

{format_instructions}
"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)
raw_structured_chain = prompt | chat | StrOutputParser()

In [38]:
print(
    prompt.format_messages(
        text=customer_review,
        format_instructions=format_instructions,
    )[0].content
)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else?
Answer true if yes, otherwise false.

delivery_days: How many days did it take for the product to arrive?
If this information is not found, output -1.

price_value: Extract any sentences about the value or price,
and output them as a JSON list of strings.

Return valid JSON with the following keys:
gift
delivery_days
price_value

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.


Return a JSON object.



In [39]:
raw_response = raw_structured_chain.invoke(
    {
        "text": customer_review,
        "format_instructions": format_instructions,
    }
)

In [40]:
print(raw_response)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}
```


In [41]:
output_dict = output_parser.invoke(raw_response)

In [42]:
output_dict

{'gift': True,
 'delivery_days': 2,
 'price_value': ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]}

In [43]:
type(output_dict)

dict

In [44]:
output_dict.get('delivery_days')

2

## Conclusion

In this notebook we explored the fundamental components used when building LLM applications with LangChain.

Key takeaways:

- LLMs can be accessed directly through APIs, but frameworks like LangChain simplify application development.
- Prompt templates help structure and reuse prompts.
- LangChain pipelines allow us to connect prompts, models, and parsers.
- Output parsers enable converting model responses into structured formats such as JSON.

These concepts form the foundation for more advanced LangChain features such as memory, agents, and tool use.

### Key Insight

An LLM application typically follows this architecture:

Prompt → Model → Output Parser → Structured Data

Understanding this pipeline is essential before building more advanced systems such as conversational agents or autonomous workflows.